my_mcp_langchain_project/
│---agent_factory
     dynamic_agentfactory.py
├── mcp_servers/
│   ├── math-mcp_server.py
│   ├── pollution-mcp_server.py
│--- config/
      setting.py
│
├── mcp_client/
│   ├── universal_mcp_client.py
│ 
 main.py

 # agent_factory/dynamic_agent_factory.py
import importlib
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool


class DynamicAgentFactory:
    """
    Dynamically creates and manages LangChain/LangGraph agents
    based on a provided configuration.

    Each agent can be linked to different MCP servers, LLM models,
    and system prompts dynamically.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.registry = {}

    async def create_agent(self, name: str):
        """
        Create and register an agent dynamically based on config.
        """
        if name not in self.config:
            raise ValueError(f"Agent '{name}' not found in configuration.")

        agent_cfg = self.config[name]
        print(f"🔧 Creating agent: {name}")
        print(f"   ↳ LLM: {agent_cfg['llm_model']}")
        print(f"   ↳ MCP Servers: {agent_cfg['mcp_servers']}")

        # 1️⃣ Load LLM
        llm = ChatOpenAI(model=agent_cfg["llm_model"], temperature=0.5)

        # 2️⃣ Dynamically import tools from MCP clients
        tools = []
        for mcp_name in agent_cfg["mcp_servers"]:
            try:
                module = importlib.import_module(f"mcp_clients.{mcp_name}_client")
                if mcp_name == "math-mcp":
                    if hasattr(module, "get_tools"):
                        mcp_tools = module.get_tools()  
                        tools.extend(mcp_tools)
                        print(f"Loaded {len(mcp_tools)} tools from {mcp_name}")
                    else:
                        print(f"No get_tools() function found in {mcp_name}_client")
                elif mcp_name == "pollution-mcp":
                    if hasattr(module, "get_pollution_tools"):
                        mcp_tools = module.get_pollution_tools()  
                        tools.extend(mcp_tools)
                        print(f"Loaded {len(mcp_tools)} tools from {mcp_name}")
                    else:
                        print(f"No get_tools() function found in {mcp_name}_client")        
            except ModuleNotFoundError:
                print(f"MCP client not found: {mcp_name}_client")

        # 3️⃣ Initialize agent
        agent = initialize_agent(
            tools=tools,
            llm=llm,
            agent="zero-shot-react-description",
            verbose=True,
        )

        # 4️⃣ Store in registry
        self.registry[name] = {
            "llm": llm,
            "tools": tools,
            "agent": agent,
            "system_prompt": agent_cfg.get("system_prompt", ""),
        }

        return agent

    async def get_agent(self, name: str):
        """
        Retrieve an agent from the registry or create it if missing.
        """
        if name not in self.registry:
            await self.create_agent(name)
        return self.registry[name]["agent"]

# math-mcp_server.py
from fastmcp.server import FastMCP
from langchain.agents import Tool

mcp = FastMCP("math-mcp")

# ----------------------------
# MCP Tools (for MCP runtime)
# ----------------------------
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


# ----------------------------
# LangChain-compatible tools
# ----------------------------
def get_tools():
    """
    Return a list of LangChain Tool objects
    that wrap the same functionality as the MCP tools.
    These can be dynamically imported and used by the agent.
    """
    return [
        Tool(
            name="add",
            func=lambda a, b: add(a, b),
            description="Add two numbers together",
        ),
        Tool(
            name="multiply",
            func=lambda a, b: multiply(a, b),
            description="Multiply two numbers together",
        ),
    ]


if __name__ == "__main__":
    print("Starting Math MCP Server...")
    mcp.run()
# pollution-mcp_server.py
from fastmcp.server import FastMCP

# Mock pollution data
POLLUTION_DATA = {
    "Delhi": "AQI 320 (Very Poor)",
    "Mumbai": "AQI 160 (Moderate)",
    "Paris": "AQI 70 (Good)"
}

# Create MCP server
mcp = FastMCP("pollution-mcp")

@mcp.tool()
def get_pollution(location: str):
    """Return pollution info for a given location"""
    return {"content": POLLUTION_DATA.get(location, "No data available")}

if __name__ == "__main__":
    mcp.run()

#setting.py
--------
import os
from dotenv import load_dotenv

# Load .env file from project root
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_MODEL = "gpt-4o"  # change to "gpt-4.1-mini" if you want cheaper


AGENT_CONFIG = {
    "agent1": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are a math expert. Use math tools wisely.",
        "mcp_servers": ["math-mcp"]
    },
    "agent3": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are an environment analyst. Use pollution data tools.",
        "mcp_servers": ["pollution-mcp"]
    }
}
# mcp_clients/universal_mcp_client.py
import asyncio
from langchain.agents import Tool

async def run_mcp_query(mcp_name: str):
    """
    Universal MCP client that dynamically loads tools from any MCP server.
    Example usage: await run_mcp_query("math_mcp")
    """
    try:
        # Dynamically import the server
        module_path = f"mcp_server.{mcp_name}_server"
        server_module = __import__(module_path, fromlist=["mcp"])
        mcp_instance = getattr(server_module, "mcp", None)

        if not mcp_instance:
            raise ValueError(f"No MCP instance found in {module_path}")

        # Collect all tools defined in the MCP server
        tools = []
        for tool_name, tool_func in mcp_instance.tools.items():
            tools.append(
                Tool(
                    name=tool_name,
                    func=tool_func,
                    description=tool_func.__doc__ or f"Tool {tool_name}",
                )
            )
        print(f"✅ Loaded {len(tools)} tools from {mcp_name}")
        return tools

    except ModuleNotFoundError:
        raise ValueError(f"MCP server not found: {mcp_name}")
    except Exception as e:
        raise RuntimeError(f"Error running MCP query for {mcp_name}: {e}")

integrate below code changes in the above code 
import asyncio
import sys
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

class MCPSession:
    """Manage a single MCP session for multiple tool calls"""

    def __init__(self, server_script_path: str):
        self.server_script_path = server_script_path
        self.session = None
        self.tools_info = []
        self._stdio_ctx = None
        self._client_session_ctx = None
        self._stdio_pair = None

    async def __aenter__(self):
        """Enter async context and initialize the session"""
        server_params = StdioServerParameters(
            command=sys.executable,
            args=[self.server_script_path]
        )

        # Open stdio transport manually
        self._stdio_ctx = stdio_client(server_params)
        self._stdio_pair = await self._stdio_ctx.__aenter__()
        read, write = self._stdio_pair

        # Start client session
        self._client_session_ctx = ClientSession(read, write)
        self.session = await self._client_session_ctx.__aenter__()

        await self.session.initialize()
        list_result = await self.session.list_tools()
        self.tools_info = list_result.tools

        print(f"✅ Connected to MCP server [{os.path.basename(self.server_script_path)}], found {len(self.tools_info)} tools")
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        """Clean up and close resources"""
        if self._client_session_ctx:
            await self._client_session_ctx.__aexit__(exc_type, exc_val, exc_tb)
            self.session = None
        if self._stdio_ctx:
            await self._stdio_ctx.__aexit__(exc_type, exc_val, exc_tb)

    async def call_tool(self, tool_name: str, **kwargs):
        """Call a tool by name"""
        if not self.session:
            raise RuntimeError("Session not initialized. Use async context manager.")

        result = await self.session.call_tool(tool_name, kwargs)
        if hasattr(result, "content") and result.content:
            return result.content[0].text
        return str(result)

    def get_tool_names(self):
        """Get list of available tool names"""
        return [tool.name for tool in self.tools_info]


# ===============================
# MULTI-SERVER DEMO
# ===============================
async def demo():
    current_dir = os.path.dirname(os.path.abspath(__file__))

    # List of MCP server scripts
    servers = [
        os.path.join(current_dir, "math-mcp_server.py"),
        os.path.join(current_dir, "pollution-mcp_server.py"),
    ]

    # Loop through each MCP server
    for server_path in servers:
        print(f"\n🔹 Connecting to server: {os.path.basename(server_path)}")

        async with MCPSession(server_path) as mcp_session:
            tool_names = mcp_session.get_tool_names()
            print(f"   Available tools: {tool_names}")

            # Dynamically handle based on server type
            if "add" in tool_names:
                result = await mcp_session.call_tool("add", a=5, b=7)
                print(f"   ➕ add(5,7) = {result}")

            if "multiply" in tool_names:
                result = await mcp_session.call_tool("multiply", a=3, b=4)
                print(f"   ✖️ multiply(3,4) = {result}")

            if "get_pollution" in tool_names:
                for city in ["Delhi", "Mumbai", "Paris", "New York"]:
                    result = await mcp_session.call_tool("get_pollution", location=city)
                    print(f"   🌆 Pollution in {city}: {result}")


if __name__ == "__main__":
    asyncio.run(demo())
